In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import Annotated, TypedDict
import operator
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3

In [3]:
# Build the graph
class State(TypedDict):
    data: Annotated[str, operator.add]   

def node_a(state: State) -> State:
    print("Node A executed")
    return {"data": "A "}

def node_b(state: State) -> State:
    print("Node B executed")
    return {"data": "B "}

def node_c(state: State) -> State:
    print("Node C executed")
    return {"data": "C "}

def node_d(state: State) -> State:
    print("Node D executed")
    return {"data": "D "}

def node_e(state: State) -> State:
    print("Node E executed")
    return {"data": "E "}

graph_builder = StateGraph(State)

graph_builder.add_node("A", node_a)
graph_builder.add_node("B", node_b)
graph_builder.add_node("C", node_c)
graph_builder.add_node("D", node_d)
graph_builder.add_node("E", node_e)

graph_builder.add_edge(START, "A")
graph_builder.add_edge("A", "B")
graph_builder.add_edge("B", "C")
graph_builder.add_edge("B", "D")
graph_builder.add_edge("C", "E")
graph_builder.add_edge("D", "E")
graph_builder.add_edge("E", END)

In [ ]:
# Create a SQLite database connection for persistent checkpoint storage
# check_same_thread=False is a SQLite setting; not related to LangGraph thread_id
conn = sqlite3.connect("checkpoints.sqlite", check_same_thread=False)

# Create a SQLite-based checkpointer
checkpointer = SqliteSaver(conn)

# Compile the graph with the SQLite checkpointer enabled
graph = graph_builder.compile(checkpointer=checkpointer)

In [4]:
# Runtime configuration with a thread_id
config = {"configurable": {"thread_id": "user-1"}}

# Invoke the graph with config
final_state = graph.invoke({"data": ""}, config=config)
print("Final state:", final_state)

Node A executed
Node B executed
Node C executed
Node D executed
Node E executed
Final state: {'data': 'A B C D E '}


In [6]:
# Retrieve the latest checkpoint for thread_id "user-1"
config= {"configurable": {"thread_id": "user-1"}}
graph.get_state(config)

StateSnapshot(values={}, next=(), config={'configurable': {'thread_id': 'user-1'}}, metadata=None, created_at=None, parent_config=None, tasks=(), interrupts=())

In [6]:
# Retrieve the full checkpoint history for this thread
list(graph.get_state_history(config))

[StateSnapshot(values={'data': 'A B C D E '}, next=(), config={'configurable': {'thread_id': 'user-1', 'checkpoint_ns': '', 'checkpoint_id': '1f0eb56a-862d-6add-8004-f69ea8e55f04'}}, metadata={'source': 'loop', 'step': 4, 'parents': {}}, created_at='2026-01-06T23:23:07.325103+00:00', parent_config={'configurable': {'thread_id': 'user-1', 'checkpoint_ns': '', 'checkpoint_id': '1f0eb56a-862b-63ea-8003-72527db3bf78'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'data': 'A B C D '}, next=('E',), config={'configurable': {'thread_id': 'user-1', 'checkpoint_ns': '', 'checkpoint_id': '1f0eb56a-862b-63ea-8003-72527db3bf78'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-01-06T23:23:07.324106+00:00', parent_config={'configurable': {'thread_id': 'user-1', 'checkpoint_ns': '', 'checkpoint_id': '1f0eb56a-8628-6cf9-8002-005a9b7162a9'}}, tasks=(PregelTask(id='e2c6f4b1-92a0-aa34-6f1a-623f49130327', name='E', path=('__pregel_pull', 'E'), error=None, interrupts=(), state

In [7]:
# Invoke the graph again using the same thread_id, execution resumes from the latest saved checkpoint
config = {"configurable": {"thread_id": "user-1"}}
final_state = graph.invoke({"data": "New "}, config=config)
print("Final state:", final_state)

Node A executed
Node B executed
Node C executed
Node D executed
Node E executed
Final state: {'data': 'A B C D E New A B C D E '}


In [8]:
# View the updated checkpoint history after resuming execution
list(graph.get_state_history(config))

[StateSnapshot(values={'data': 'A B C D E New A B C D E '}, next=(), config={'configurable': {'thread_id': 'user-1', 'checkpoint_ns': '', 'checkpoint_id': '1f0eb56e-aaca-621d-800a-03f429cee66a'}}, metadata={'source': 'loop', 'step': 10, 'parents': {}}, created_at='2026-01-06T23:24:58.538242+00:00', parent_config={'configurable': {'thread_id': 'user-1', 'checkpoint_ns': '', 'checkpoint_id': '1f0eb56e-aac7-6b1f-8009-6f310a46d5d5'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'data': 'A B C D E New A B C D '}, next=('E',), config={'configurable': {'thread_id': 'user-1', 'checkpoint_ns': '', 'checkpoint_id': '1f0eb56e-aac7-6b1f-8009-6f310a46d5d5'}}, metadata={'source': 'loop', 'step': 9, 'parents': {}}, created_at='2026-01-06T23:24:58.537244+00:00', parent_config={'configurable': {'thread_id': 'user-1', 'checkpoint_ns': '', 'checkpoint_id': '1f0eb56e-aac0-6643-8008-795f7f21be3c'}}, tasks=(PregelTask(id='7f1f6e3e-400e-e1ee-f03d-702d9fd810eb', name='E', path=('__pregel_pull', 'E'), err

In [9]:
# Invoke the graph with a different thread_id, this creates a new, independent workflow execution
final_state = graph.invoke({"data": "222 "}, config={"configurable": {"thread_id": "user-2"}})

Node A executed
Node B executed
Node C executed
Node D executed
Node E executed
